In [7]:
import os
from IPython.display import display, HTML
import time
import datetime
from datetime import date, datetime
import matplotlib.pyplot as plt
import numpy as np
import sys
import getpass
import json
import matplotlib.pyplot as plt
import numpy as np
from pprint import pprint
from pyproj import Transformer
import rasterio
from rasterio.mask import mask
from rasterio.windows import from_bounds
import requests
from shapely.geometry import Polygon, mapping, shape
from requests.auth import HTTPBasicAuth
if os.environ.get('PL_API_KEY', ''):
    API_KEY = os.environ.get('PL_API_KEY', '')
else:
    API_KEY = '1c61ae19230448e19b5bc359cbf1232e'
    
session = requests.Session()
session.auth = (API_KEY, "")


In [45]:
display(HTML("""cloud coverage in decimal)"""))
cc = float(input().strip())
display(HTML("""<b>Input date (ex. 2025-05-27)"""))
input_date = input().strip()

0.1


2025-05-27


In [ ]:
# #Create directories for all locations in local directory
json_dir = 'albemarle_geojsons/'
image_ids_=set()
for file in os.listdir(json_dir):
    folder_name = file.split('.')[0]
#     os.mkdir(folder_name)
    with open(os.path.join(json_dir,file), 'r') as file:
        data = json.load(file)
        
        date_obj = datetime.strptime(input_date, "%Y-%m-%d")
        date_2_days_prior = date_obj - timedelta(days=2)
        date_lte = date_obj.strftime('%Y-%m-%dT') + '00:00:00.000Z'
        date_gte = date_2_days_prior.strftime('%Y-%m-%dT') + '00:00:00.000Z'

        geojson_geometry = {
            "type": "Polygon",
            "coordinates": data['features'][0]['geometry']['coordinates']}

        # get images that overlap with our AOI 
        geometry_filter = {
          "type": "GeometryFilter",
          "field_name": "geometry",
          "config": geojson_geometry}

        cloud_cover_filter = {
          "type": "RangeFilter",
          "field_name": "cloud_cover",
          "config": {
            "lte": cc}}

        quality_filter = {
           "type":"StringInFilter",
           "field_name":"quality_category",
           "config":[
              "standard"]}
        # get images acquired within a date range
        date_range_filter = {
          "type": "DateRangeFilter",
          "field_name": "acquired",
          "config": {
            "gte": date_gte,
            "lte": date_lte}}

        # combine our geo, date, cloud filters
        combined_filter = {
          "type": "AndFilter",
          "config": [geometry_filter, date_range_filter, cloud_cover_filter, quality_filter]}



        #Edit planet item type
        item_type = "PSScene"

        # API request object
        search_request = {
          "item_types": [item_type], 
          "filter": combined_filter
        }
        # fire off the POST request
        search_result = \
          requests.post(
            'https://api.planet.com/data/v1/quick-search',
            auth=HTTPBasicAuth(API_KEY, ''),
            json=search_request)

        results = search_result.json()
        feat = results['features'][0]
        item_geom = feat['geometry']
        image_ids = list({feature['id'] for feature in results['features']})

        image_ids = [feature['id'] for feature in results['features']]
#     image_ids_.add(image_ids)
#         print(image_ids)

In [33]:
import datetime
from datetime import datetime, timedelta



In [56]:
image_ids

['20250525_161002_74_250a', '20250525_161000_50_250a']

NameError: name 'data' is not defined